# nto_final_strong_solution_(1) — NTO Lost Positives (Strong)

Пайплайн: incident-aware validation + multi-retrieval candidates + ranker ensemble.


In [ ]:
# !pip install implicit catboost scikit-learn -q
import os, gc, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import sparse
from collections import defaultdict

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

import implicit
from catboost import CatBoostRanker, Pool
from tqdm.auto import tqdm

SEED = 42
np.random.seed(SEED)
TOP_K = 20
CANDS_PER_USER = 320
ALS_FACTORS = 256
ALS_ITERS = 45
ALS_REG = 0.025
KNN_K = 240

INCIDENT_START = pd.Timestamp("2025-10-01 00:00:00")
INCIDENT_END   = pd.Timestamp("2025-11-01 00:00:00")
HIDDEN_SHARE = 0.20
DATA_DIR = "./data"


In [ ]:
def reduce_mem(df: pd.DataFrame) -> pd.DataFrame:
    for c in df.columns:
        if pd.api.types.is_integer_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], downcast="integer")
        elif pd.api.types.is_float_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], downcast="float")
    return df

interactions = pd.read_csv(f"{DATA_DIR}/interactions.csv", parse_dates=["event_ts"])
targets = pd.read_csv(f"{DATA_DIR}/targets.csv")
editions = pd.read_csv(f"{DATA_DIR}/editions.csv")
book_genres = pd.read_csv(f"{DATA_DIR}/book_genres.csv")
users = pd.read_csv(f"{DATA_DIR}/users.csv")

for name in ["interactions", "targets", "editions", "book_genres", "users"]:
    globals()[name] = reduce_mem(globals()[name])

interactions = interactions[interactions.event_type.isin([1, 2])].copy()
interactions["w"] = np.where(interactions.event_type.eq(2), 1.45, 1.0).astype("float32")

max_ts = interactions.event_ts.max()
age_days = (max_ts - interactions.event_ts).dt.days.clip(lower=0).astype("float32")
interactions["time_decay"] = np.exp(-age_days / 45.0).astype("float32")
interactions["w_time"] = (interactions["w"] * (0.65 + 0.7 * interactions["time_decay"]))
interactions["w_time"] = interactions["w_time"].astype("float32")

print(interactions.shape, targets.shape, editions.shape)


In [ ]:
def make_validation_split_incident(interactions: pd.DataFrame, hidden_share: float = HIDDEN_SHARE, seed: int = SEED):
    rng = np.random.default_rng(seed)
    inc = interactions[(interactions.event_ts >= INCIDENT_START) & (interactions.event_ts < INCIDENT_END)].copy()
    hist = interactions[interactions.event_ts < INCIDENT_START].copy()

    pos_pairs = inc[["user_id", "edition_id"]].drop_duplicates().reset_index(drop=True)
    masked_idx = []
    for _, idx in pos_pairs.groupby("user_id").groups.items():
        idx = np.array(list(idx), dtype=np.int64)
        m = max(1, int(np.ceil(len(idx) * hidden_share)))
        masked_idx.extend(rng.choice(idx, size=min(m, len(idx)), replace=False))

    hidden_pairs = pos_pairs.loc[masked_idx].copy()
    hidden_pairs["target"] = 1
    hidden_set = set(map(tuple, hidden_pairs[["user_id", "edition_id"]].to_numpy()))

    inc_obs = inc[~inc[["user_id", "edition_id"]].apply(tuple, axis=1).isin(hidden_set)]
    train_obs = pd.concat([hist, inc_obs], ignore_index=True)
    return train_obs, hidden_pairs


def ndcg_at_k(pred: pd.DataFrame, gt: pd.DataFrame, k: int = 20) -> float:
    gt_map = gt.groupby("user_id")["edition_id"].apply(set).to_dict()
    scores = []
    for u, g in pred.groupby("user_id"):
        truth = gt_map.get(u, set())
        if not truth:
            continue
        rel = np.array([1.0 if e in truth else 0.0 for e in g.sort_values("rank").edition_id.values[:k]], dtype="float32")
        discounts = 1.0 / np.log2(np.arange(2, rel.size + 2))
        dcg = float((rel * discounts).sum())
        m = min(len(truth), k)
        idcg = float((np.ones(m) * (1.0 / np.log2(np.arange(2, m + 2)))).sum())
        scores.append(dcg / idcg if idcg > 0 else 0.0)
    return float(np.mean(scores)) if scores else 0.0


In [ ]:
def fit_index(obs: pd.DataFrame):
    users_idx = pd.Index(obs.user_id.unique(), name="user_id")
    items_idx = pd.Index(editions.edition_id.unique(), name="edition_id")
    u2i = {u: i for i, u in enumerate(users_idx)}
    e2i = {e: i for i, e in enumerate(items_idx)}
    return users_idx, items_idx, u2i, e2i

def build_ui(obs: pd.DataFrame, u2i: dict, e2i: dict, w_col: str = "w_time"):
    x = obs[obs.user_id.isin(u2i) & obs.edition_id.isin(e2i)]
    r = x.user_id.map(u2i).to_numpy(); c = x.edition_id.map(e2i).to_numpy()
    w = x[w_col].astype("float32").to_numpy()
    return sparse.csr_matrix((w, (r, c)), shape=(len(u2i), len(e2i)), dtype="float32")

def train_retrievers(ui: sparse.csr_matrix):
    als = implicit.als.AlternatingLeastSquares(
        factors=ALS_FACTORS, iterations=ALS_ITERS, regularization=ALS_REG,
        alpha=40, random_state=SEED, calculate_training_loss=False,
    )
    als.fit(ui.T.tocsr())
    knn = implicit.nearest_neighbours.CosineRecommender(K=KNN_K)
    knn.fit(ui.T.tocsr())
    return als, knn


In [ ]:
def build_text_embeddings(editions: pd.DataFrame):
    txt = (editions.title.fillna("") + " " + editions.description.fillna("")).str.lower()
    tfidf = TfidfVectorizer(max_features=150000, ngram_range=(1, 2), min_df=3)
    X = tfidf.fit_transform(txt)
    svd = TruncatedSVD(n_components=192, random_state=SEED)
    E = normalize(svd.fit_transform(X).astype("float32"))
    emb = pd.DataFrame({"edition_id": editions.edition_id.values})
    for i in range(E.shape[1]):
        emb[f"txt_{i}"] = E[:, i]
    return emb.set_index("edition_id"), tfidf, svd

def user_text_profiles(obs: pd.DataFrame, txt_item_emb: pd.DataFrame):
    tmp = obs[["user_id", "edition_id", "w_time"]].merge(txt_item_emb.reset_index(), on="edition_id", how="left").fillna(0)
    txt_cols = [c for c in tmp.columns if c.startswith("txt_")]
    tmp[txt_cols] = tmp[txt_cols].mul(tmp["w_time"], axis=0)
    return tmp.groupby("user_id")[txt_cols].mean().astype("float32")

def make_pop_tables(obs: pd.DataFrame):
    global_pop = obs.groupby("edition_id").size().astype("float32")
    last_14 = obs[obs.event_ts >= (obs.event_ts.max() - pd.Timedelta(days=14))]
    last_45 = obs[obs.event_ts >= (obs.event_ts.max() - pd.Timedelta(days=45))]
    prev_45 = obs[(obs.event_ts < (obs.event_ts.max() - pd.Timedelta(days=45))) & (obs.event_ts >= (obs.event_ts.max() - pd.Timedelta(days=90)))]
    pop14 = last_14.groupby("edition_id").size().astype("float32")
    pop45 = last_45.groupby("edition_id").size().astype("float32")
    pop_prev45 = prev_45.groupby("edition_id").size().astype("float32")
    trend = (pop45 / (pop_prev45 + 1.0)).astype("float32")
    return {
        "global_pop": global_pop, "pop14": pop14, "pop45": pop45, "pop_prev45": pop_prev45, "trend": trend,
        "hot_items": pop14.sort_values(ascending=False).head(1200).index.to_numpy(),
        "pop_items": global_pop.sort_values(ascending=False).head(1200).index.to_numpy(),
        "trend_items": trend.sort_values(ascending=False).head(1200).index.to_numpy(),
    }


In [ ]:
def build_side_maps(obs: pd.DataFrame, editions: pd.DataFrame, book_genres: pd.DataFrame):
    obs_e = obs[["user_id", "edition_id"]].merge(editions[["edition_id", "author_id", "book_id"]], on="edition_id", how="left")
    bg = editions[["edition_id", "book_id"]].merge(book_genres, on="book_id", how="left")
    user_seen = obs.groupby("user_id")["edition_id"].agg(set).to_dict()
    user_auth = obs_e.groupby("user_id")["author_id"].agg(lambda x: set(x.dropna())).to_dict()
    ug = obs[["user_id", "edition_id"]].merge(bg[["edition_id", "genre_id"]], on="edition_id", how="left")
    user_genres = ug.groupby("user_id")["genre_id"].agg(lambda x: set(x.dropna())).to_dict()

    top_items = obs.groupby("edition_id").size().sort_values(ascending=False).head(18000).index
    sub = obs[obs.edition_id.isin(top_items)][["user_id", "edition_id"]].drop_duplicates()
    co_map = defaultdict(list)
    for items in sub.groupby("user_id")["edition_id"].apply(list):
        limited = items[:40]
        for i in range(len(limited)):
            for j in range(i + 1, len(limited)):
                a, b = limited[i], limited[j]
                co_map[a].append(b); co_map[b].append(a)
    co_neighbors = {}
    for e, lst in co_map.items():
        if lst:
            co_neighbors[e] = list(pd.Series(lst).value_counts().head(40).index.astype(int))
    return user_seen, user_auth, user_genres, co_neighbors


def generate_candidates(user_list, obs, users_idx, items_idx, u2i, e2i, als, knn, txt_item_emb, txt_user_emb, editions, book_genres, cands_n=CANDS_PER_USER):
    i2e = np.array(items_idx)
    ui = build_ui(obs, u2i, e2i, w_col="w_time")
    pop_tables = make_pop_tables(obs)
    user_seen, user_auth, user_genres, co_neighbors = build_side_maps(obs, editions, book_genres)
    bg = editions[["edition_id", "book_id"]].merge(book_genres, on="book_id", how="left")
    genre2ed = bg.groupby("genre_id")["edition_id"].apply(lambda x: list(pd.unique(x.dropna()))).to_dict()
    txt_mat = txt_item_emb.to_numpy(dtype="float32"); txt_ids = txt_item_emb.index.to_numpy()

    rows = []
    for u in tqdm(user_list):
        seen = user_seen.get(u, set()); cand = {}
        def push(eid, score, src):
            if eid in seen: return
            prev = cand.get(eid)
            if (prev is None) or (score > prev[0]):
                cand[eid] = [float(score), src]

        if u in u2i:
            uid = u2i[u]
            rec_i, rec_s = als.recommend(uid, ui[uid], N=220, filter_already_liked_items=False)
            for iid, s in zip(rec_i, rec_s): push(int(i2e[iid]), float(s), "als")

        recent = obs[obs.user_id.eq(u)].sort_values("event_ts").edition_id.values[-35:]
        for e in recent:
            if e in e2i:
                sim_i, sim_s = knn.similar_items(e2i[e], N=45)
                for iid, s in zip(sim_i, sim_s): push(int(i2e[iid]), float(s) * 0.85, "knn")
                for ee in co_neighbors.get(int(e), [])[:25]: push(int(ee), 0.35, "co")

        if u in txt_user_emb.index:
            uv = txt_user_emb.loc[u].to_numpy(dtype="float32")
            sims = txt_mat @ uv
            top_idx = np.argpartition(-sims, 180)[:180]
            for j in top_idx: push(int(txt_ids[j]), float(sims[j]) * 0.9, "txt")

        for a in list(user_auth.get(u, set()))[:30]:
            if pd.isna(a): continue
            for e in editions.loc[editions.author_id.eq(a), "edition_id"].values[:120]: push(int(e), 0.22, "author")

        for g in list(user_genres.get(u, set()))[:10]:
            for e in genre2ed.get(g, [])[:180]: push(int(e), 0.18, "genre")

        for e in pop_tables["hot_items"][:350]: push(int(e), 0.15, "hot")
        for e in pop_tables["trend_items"][:350]: push(int(e), 0.13, "trend")
        for e in pop_tables["pop_items"][:350]: push(int(e), 0.10, "pop")

        cand_items = sorted(cand.items(), key=lambda x: x[1][0], reverse=True)[:cands_n]
        for eid, (score, src) in cand_items:
            rows.append((u, int(eid), float(score), src))

    cands = pd.DataFrame(rows, columns=["user_id", "edition_id", "base_score", "source"])
    return cands, pop_tables


In [ ]:
def build_features(cands, obs, editions, users, book_genres, als, u2i, e2i, pop_tables):
    df = cands.copy()
    df = df.merge(editions[["edition_id", "author_id", "publication_year", "language_id", "publisher_id", "book_id"]], on="edition_id", how="left")
    df = df.merge(users[["user_id", "gender", "age"]], on="user_id", how="left")

    item_pop = pop_tables["global_pop"].rename("item_pop")
    item_hot = pop_tables["pop14"].rename("item_hot")
    item_45 = pop_tables["pop45"].rename("item_45")
    item_prev_45 = pop_tables["pop_prev45"].rename("item_prev_45")
    item_trend = pop_tables["trend"].rename("item_trend")

    user_pop = obs.groupby("user_id").size().rename("user_pop").astype("float32")
    user_recent = obs[obs.event_ts >= (obs.event_ts.max() - pd.Timedelta(days=30))].groupby("user_id").size().rename("user_recent").astype("float32")

    for s, key in [(item_pop, "edition_id"), (item_hot, "edition_id"), (item_45, "edition_id"), (item_prev_45, "edition_id"), (item_trend, "edition_id")]:
        df = df.merge(s, on=key, how="left")
    df = df.merge(user_pop, on="user_id", how="left").merge(user_recent, on="user_id", how="left")

    for c in ["item_pop", "item_hot", "item_45", "item_prev_45", "item_trend", "user_pop", "user_recent"]:
        df[c] = df[c].fillna(0).astype("float32")

    df["item_hot_ratio"] = (df["item_hot"] / (df["item_45"] + 1.0)).astype("float32")
    df["item_trend_ratio"] = (df["item_45"] / (df["item_prev_45"] + 1.0)).astype("float32")
    df["user_hot_ratio"] = (df["user_recent"] / (df["user_pop"] + 1.0)).astype("float32")
    df["cross_hot"] = (df["user_hot_ratio"] * df["item_hot_ratio"]).astype("float32")

    bg = editions[["edition_id", "book_id"]].merge(book_genres, on="book_id", how="left")
    ug = obs[["user_id", "edition_id"]].merge(bg[["edition_id", "genre_id"]], on="edition_id", how="left")
    ug_cnt = ug.groupby(["user_id", "genre_id"]).size().rename("ug_cnt").reset_index()
    df = df.merge(bg[["edition_id", "genre_id"]], on="edition_id", how="left")
    df = df.merge(ug_cnt, on=["user_id", "genre_id"], how="left")
    df["ug_cnt"] = df["ug_cnt"].fillna(0).astype("float32")

    ua = obs[["user_id", "edition_id"]].merge(editions[["edition_id", "author_id"]], on="edition_id", how="left").groupby(["user_id", "author_id"]).size().rename("ua_cnt").reset_index()
    ub = obs[["user_id", "edition_id"]].merge(editions[["edition_id", "book_id"]], on="edition_id", how="left").groupby(["user_id", "book_id"]).size().rename("ub_cnt").reset_index()
    df = df.merge(ua, on=["user_id", "author_id"], how="left").merge(ub, on=["user_id", "book_id"], how="left")
    df["ua_cnt"] = df["ua_cnt"].fillna(0).astype("float32")
    df["ub_cnt"] = df["ub_cnt"].fillna(0).astype("float32")

    df["als_dot"] = 0.0
    idx = df[df.user_id.isin(u2i) & df.edition_id.isin(e2i)].index
    if len(idx) > 0:
        uids = df.loc[idx, "user_id"].map(u2i).to_numpy(); iids = df.loc[idx, "edition_id"].map(e2i).to_numpy()
        df.loc[idx, "als_dot"] = (als.user_factors[uids] * als.item_factors[iids]).sum(axis=1)

    df = pd.concat([df, pd.get_dummies(df["source"], prefix="src", dtype="int8")], axis=1)
    df["age"] = df["age"].fillna(-1); df["publication_year"] = df["publication_year"].fillna(-1)
    return df


In [ ]:
def train_ranker_ensemble(train_df: pd.DataFrame, y: np.ndarray, seed: int = SEED):
    train_df = train_df.sort_values("user_id").reset_index(drop=True)
    y = y[train_df.index]
    feat_cols = [c for c in train_df.columns if c not in ["user_id", "edition_id", "target"]]
    cat_cols = [c for c in ["gender", "language_id", "publisher_id", "author_id"] if c in feat_cols]
    cat_idx = [feat_cols.index(c) for c in cat_cols]
    pool = Pool(train_df[feat_cols], label=y, group_id=train_df["user_id"], cat_features=cat_idx)

    models = []
    cfgs = [
        dict(loss_function="YetiRankPairwise", iterations=1300, learning_rate=0.045, depth=8, random_seed=seed, verbose=250),
        dict(loss_function="PairLogitPairwise", iterations=1100, learning_rate=0.055, depth=7, random_seed=seed + 7, verbose=250),
        dict(loss_function="YetiRank", iterations=900, learning_rate=0.06, depth=8, random_seed=seed + 21, verbose=250),
    ]
    for cfg in cfgs:
        m = CatBoostRanker(**cfg); m.fit(pool); models.append(m)
    return models, feat_cols


def infer_rank(cands_df: pd.DataFrame, models, feat_cols, topk: int = 20):
    score = np.mean(np.vstack([m.predict(cands_df[feat_cols]) for m in models]), axis=0)
    out = cands_df[["user_id", "edition_id"]].copy()
    out["score"] = score
    out = out.sort_values(["user_id", "score"], ascending=[True, False])
    out["rank"] = out.groupby("user_id").cumcount() + 1
    return out[out["rank"] <= topk][["user_id", "edition_id", "rank"]]


def enforce_topk(sub: pd.DataFrame, user_ids: np.ndarray, fallback_items: np.ndarray, k: int = 20):
    rows = []; sub_map = {u: g.sort_values("rank")["edition_id"].tolist() for u, g in sub.groupby("user_id")}
    for u in user_ids:
        lst = sub_map.get(u, []); seen = set(lst)
        if len(lst) < k:
            for e in fallback_items:
                ee = int(e)
                if ee not in seen:
                    lst.append(ee); seen.add(ee)
                if len(lst) >= k: break
        for r, e in enumerate(lst[:k], 1): rows.append((int(u), int(e), int(r)))
    return pd.DataFrame(rows, columns=["user_id", "edition_id", "rank"])


In [ ]:
# ===== Validation =====
obs_train, hidden_gt = make_validation_split_incident(interactions, hidden_share=HIDDEN_SHARE, seed=SEED)
users_idx, items_idx, u2i, e2i = fit_index(obs_train)
ui = build_ui(obs_train, u2i, e2i, w_col="w_time")
als, knn = train_retrievers(ui)

txt_item_emb, tfidf, svd = build_text_embeddings(editions)
txt_user_emb = user_text_profiles(obs_train, txt_item_emb)

val_users = hidden_gt.user_id.unique()
val_cands, val_pop = generate_candidates(val_users, obs_train, users_idx, items_idx, u2i, e2i, als, knn, txt_item_emb, txt_user_emb, editions, book_genres, cands_n=CANDS_PER_USER)
val_df = build_features(val_cands, obs_train, editions, users, book_genres, als, u2i, e2i, val_pop)
val_pairs = hidden_gt[["user_id", "edition_id"]].drop_duplicates().assign(target=1)
val_df = val_df.merge(val_pairs, on=["user_id", "edition_id"], how="left")
val_df["target"] = val_df["target"].fillna(0).astype("int8")

models, feat_cols = train_ranker_ensemble(val_df, val_df["target"].to_numpy(), seed=SEED)
val_pred = infer_rank(val_df.drop(columns=["target"]), models, feat_cols, topk=TOP_K)
print("VAL NDCG@20 =", ndcg_at_k(val_pred, hidden_gt[["user_id", "edition_id"]], k=20))


In [ ]:
# ===== Full train + submission =====
full_obs = interactions.copy()
users_idx, items_idx, u2i, e2i = fit_index(full_obs)
ui = build_ui(full_obs, u2i, e2i, w_col="w_time")
als, knn = train_retrievers(ui)

txt_item_emb, tfidf, svd = build_text_embeddings(editions)
txt_user_emb = user_text_profiles(full_obs, txt_item_emb)

test_users = targets.user_id.unique()
test_cands, test_pop = generate_candidates(test_users, full_obs, users_idx, items_idx, u2i, e2i, als, knn, txt_item_emb, txt_user_emb, editions, book_genres, cands_n=CANDS_PER_USER)
test_df = build_features(test_cands, full_obs, editions, users, book_genres, als, u2i, e2i, test_pop)
sub = infer_rank(test_df, models, feat_cols, topk=TOP_K)
sub = enforce_topk(sub, test_users, fallback_items=test_pop["hot_items"][:600], k=TOP_K)
sub.to_csv("submission.csv", index=False)
print(sub.shape)
sub.head(10)
